# Walmart Sales Forecasting

## Business Problem

Walmart wants to predict future weekly sales for every store and department.

Accurate sales forecasts help Walmart:

- Maintain optimal inventory levels
- Reduce stock shortages and overstocking
- Schedule employees efficiently
- Improve supply chain planning
- Support promotional planning during holidays

## Project Goal

Develop machine learning models capable of forecasting weekly sales using historical sales, store characteristics, and external economic factors.

# 01 - Data Understanding

## Business Objective

The objective of this notebook is to gain a comprehensive understanding of the Walmart Sales Forecasting dataset. This includes examining the available data sources, identifying the structure and relationships between datasets, assessing data quality, and understanding the business context behind each feature. A solid understanding of the data is essential for building accurate and reliable sales forecasting models.

## Analytical Objectives

- Load all project datasets into Python.
- Understand the purpose of each dataset.
- Explore the dimensions (rows and columns) of each table.
- Inspect column names and data types.
- Identify missing values and duplicate records.
- Generate descriptive statistics for numerical features.
- Understand categorical variables and their unique values.
- Explore the relationship between sales, stores, and external factors.
- Identify the primary and foreign keys used to join datasets.
- Document initial observations and potential data quality issues.

## Datasets

The project uses four datasets:

| Dataset | Description |
|---------|-------------|
| train.csv | Historical weekly sales used for model training |
| test.csv | Future records where weekly sales must be predicted |
| features.csv | Store-level external features such as temperature, fuel price, CPI, unemployment, markdowns, holidays, etc. |
| stores.csv | Store information including store type and size |

## Expected Outcome

By the end of this notebook, we will have:

- A complete understanding of the dataset structure.
- Knowledge of how all datasets are related.
- Identification of missing values and inconsistencies.
- Documentation of important business features.
- A clear roadmap for the data cleaning and feature engineering stages.

#### Import Libraries

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

#### Load Dataset

In [19]:
train = pd.read_csv("../data/raw/train.csv")

features = pd.read_csv("../data/raw/features.csv")

stores = pd.read_csv("../data/raw/stores.csv")

test = pd.read_csv("../data/raw/test.csv")

#### Preview Dataset

In [20]:
train.head(5)

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [21]:
features.head(5)

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [22]:
stores.head(5)

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [23]:
test.head(5)

,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


#### Dataset Shape

In [24]:
print("Train Shape :", train.shape)
print("Features Shape :", features.shape)
print("Stores Shape :", stores.shape)
print("Test Shape :", test.shape)

Train Shape : (421570, 5)
Features Shape : (8190, 12)
Stores Shape : (45, 3)
Test Shape : (115064, 4)


#### Dataset Information

In [25]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  str    
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), str(1)
memory usage: 13.3 MB


In [26]:
features.info()

<class 'pandas.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         8190 non-null   int64  
 1   Date          8190 non-null   str    
 2   Temperature   8190 non-null   float64
 3   Fuel_Price    8190 non-null   float64
 4   MarkDown1     4032 non-null   float64
 5   MarkDown2     2921 non-null   float64
 6   MarkDown3     3613 non-null   float64
 7   MarkDown4     3464 non-null   float64
 8   MarkDown5     4050 non-null   float64
 9   CPI           7605 non-null   float64
 10  Unemployment  7605 non-null   float64
 11  IsHoliday     8190 non-null   bool   
dtypes: bool(1), float64(9), int64(1), str(1)
memory usage: 712.0 KB


In [27]:
stores.info()

<class 'pandas.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Store   45 non-null     int64
 1   Type    45 non-null     str  
 2   Size    45 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.2 KB


In [28]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 115064 entries, 0 to 115063
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   Store      115064 non-null  int64
 1   Dept       115064 non-null  int64
 2   Date       115064 non-null  str  
 3   IsHoliday  115064 non-null  bool 
dtypes: bool(1), int64(2), str(1)
memory usage: 2.7 MB


#### Check Missing Values

In [29]:
train.isnull().sum()

Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
dtype: int64

In [30]:
features.isnull().sum()

Store              0
Date               0
Temperature        0
Fuel_Price         0
MarkDown1       4158
MarkDown2       5269
MarkDown3       4577
MarkDown4       4726
MarkDown5       4140
CPI              585
Unemployment     585
IsHoliday          0
dtype: int64

In [31]:
stores.isnull().sum()

Store    0
Type     0
Size     0
dtype: int64

In [32]:
test.isnull().sum()

Store        0
Dept         0
Date         0
IsHoliday    0
dtype: int64

#### Check Duplicate Rows

In [34]:
train.duplicated().sum()

np.int64(0)

In [35]:
features.duplicated().sum()

np.int64(0)

In [36]:
stores.duplicated().sum()

np.int64(0)

In [37]:
test.duplicated().sum()

np.int64(0)

#### Column Names

In [38]:
train.columns

Index(['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday'], dtype='str')

In [39]:
features.columns

Index(['Store', 'Date', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2',
       'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment',
       'IsHoliday'],
      dtype='str')

In [40]:
stores.columns

Index(['Store', 'Type', 'Size'], dtype='str')

In [41]:
test.columns

Index(['Store', 'Dept', 'Date', 'IsHoliday'], dtype='str')

#### Unique Values in Categorical Columns

In [61]:
train["Store"].nunique()

45

In [43]:
train["Dept"].nunique()

81

In [ ]:
train["IsHoliday"].value_counts()

2

In [54]:
stores["Type"].value_counts()

Type
A    22
B    17
C     6
Name: count, dtype: int64

#### Date Range

In [47]:
train["Date"]=pd.to_datetime(train["Date"])
print("Minimum Date:", min(train["Date"]))
print("Maximum Date:", max(train["Date"]))

Minimum Date: 2010-02-05 00:00:00
Maximum Date: 2012-10-26 00:00:00


#### Target Variable Statistics

In [48]:
train["Weekly_Sales"].describe()

count    421570.000000
mean      15981.258123
std       22711.183519
min       -4988.940000
25%        2079.650000
50%        7612.030000
75%       20205.852500
max      693099.360000
Name: Weekly_Sales, dtype: float64

#### Descriptive Statistics

In [49]:
train.describe(include="all")

,Store,Dept,Date,Weekly_Sales,IsHoliday
count,421570.000000,421570.000000,421570,421570.000000,421570
unique,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,NaN,False
freq,NaN,NaN,NaN,NaN,391909
mean,22.200546,44.260317,2011-06-18 08:30:31.963375,15981.258123,NaN
min,1.000000,1.000000,2010-02-05 00:00:00,-4988.940000,NaN
25%,11.000000,18.000000,2010-10-08 00:00:00,2079.650000,NaN
50%,22.000000,37.000000,2011-06-17 00:00:00,7612.030000,NaN
75%,33.000000,74.000000,2012-02-24 00:00:00,20205.852500,NaN
max,45.000000,99.000000,2012-10-26 00:00:00,693099.360000,NaN


In [50]:
features.describe(include="all")

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
count,8190.000000,8190,8190.000000,8190.000000,4032.000000,2921.000000,3613.000000,3464.000000,4050.000000,7605.000000,7605.000000,8190
unique,NaN,182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,2010-02-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
freq,NaN,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7605
mean,23.000000,NaN,59.356198,3.405992,7032.371786,3384.176594,1760.100180,3292.935886,4132.216422,172.460809,7.826821,NaN
std,12.987966,NaN,18.678607,0.431337,9262.747448,8793.583016,11276.462208,6792.329861,13086.690278,39.738346,1.877259,NaN
min,1.000000,NaN,-7.290000,2.472000,-2781.450000,-265.760000,-179.260000,0.220000,-185.170000,126.064000,3.684000,NaN
25%,12.000000,NaN,45.902500,3.041000,1577.532500,68.880000,6.600000,304.687500,1440.827500,132.364839,6.634000,NaN
50%,23.000000,NaN,60.710000,3.513000,4743.580000,364.570000,36.260000,1176.425000,2727.135000,182.764003,7.806000,NaN
75%,34.000000,NaN,73.880000,3.743000,8923.310000,2153.350000,163.150000,3310.007500,4832.555000,213.932412,8.567000,NaN


In [51]:
stores.describe(include="all")

,Store,Type,Size
count,45.000000,45,45.000000
unique,NaN,3,NaN
top,NaN,A,NaN
freq,NaN,22,NaN
mean,23.000000,NaN,130287.600000
std,13.133926,NaN,63825.271991
min,1.000000,NaN,34875.000000
25%,12.000000,NaN,70713.000000
50%,23.000000,NaN,126512.000000
75%,34.000000,NaN,202307.000000


In [52]:
test.describe(include="all")

,Store,Dept,Date,IsHoliday
count,115064.000000,115064.000000,115064,115064
unique,NaN,NaN,39,2
top,NaN,NaN,2012-12-21,False
freq,NaN,NaN,3002,106136
mean,22.238207,44.339524,NaN,NaN
std,12.809930,30.656410,NaN,NaN
min,1.000000,1.000000,NaN,NaN
25%,11.000000,18.000000,NaN,NaN
50%,22.000000,37.000000,NaN,NaN
75%,33.000000,74.000000,NaN,NaN


#### Verify Join Keys

In [63]:
# Check whether every Store in train exists in stores
print(train["Store"].isin(stores["Store"]).all())

True


In [64]:
# Check whether every Store in test exists in stores
print(test["Store"].isin(stores["Store"]).all())

True


In [69]:
features["Date"]=pd.to_datetime(features["Date"])

In [72]:
test["Date"]=pd.to_datetime(test["Date"])

In [70]:
# Verify merge between train and features
train_features = train.merge(
    features,
    on=["Store", "Date", "IsHoliday"],
    how="left"
)

print(train_features.shape)

(421570, 14)


In [73]:
# Verify merge between test and features
test_features = test.merge(
    features,
    on=["Store", "Date", "IsHoliday"],
    how="left"
)

print(test_features.shape)

(115064, 13)


In [74]:
train_final = (
    train
    .merge(features, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(stores, on="Store", how="left")
)

print(train_final.shape)

(421570, 16)


In [75]:
test_final = (
    test
    .merge(features, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(stores, on="Store", how="left")
)

print(test_final.shape)

(115064, 15)


In [76]:
print("Train Stores Missing:", train["Store"].isin(stores["Store"]).sum() != len(train))
print("Test Stores Missing :", test["Store"].isin(stores["Store"]).sum() != len(test))

Train Stores Missing: False
Test Stores Missing : False


In [77]:
# Count missing values in a feature column after merge
print(train_features["Temperature"].isna().sum())
print(test_features["Temperature"].isna().sum())

0
0


#### Dataset Relationships

-Store links stores to train.

-Store + Date links features to train.

-test uses the same relationships as train.

#### Key Observations

- The Walmart Sales Forecasting dataset consists of **four interconnected datasets**: historical sales, external features, store information, and future prediction records.
- The **training dataset** contains **421,570 historical weekly sales records** across **45 stores** and **81 departments**, providing a robust foundation for forecasting.
- The historical data spans **February 2010 to October 2012**, covering nearly three years of sales activity and capturing seasonal and holiday patterns.
- The **test dataset** contains **115,064 future observations** without the `Weekly_Sales` column, which will be predicted by the forecasting models.
- The **features dataset** enriches the sales data with weather conditions, fuel prices, promotional markdowns, holidays, and economic indicators such as CPI and unemployment.
- The **stores dataset** contains metadata for **45 Walmart stores**, including store type and physical size, enabling store-level analysis.
- Missing values are present **only in the `features.csv` dataset**, primarily in the promotional `MarkDown1–MarkDown5` columns and, to a lesser extent, the `CPI` and `Unemployment` columns.
- No duplicate records were found in any of the four datasets, indicating strong data integrity.
- The `Date` column is stored as a string in multiple datasets and must be converted to the **datetime** data type during preprocessing.
- Weekly sales range from **-$4,988.94** to **$693,099.36**, indicating substantial variation across stores, departments, and time periods.
- The sales distribution is **right-skewed**, with a relatively small number of exceptionally high sales observations increasing the overall average.
- Approximately **93%** of the observations correspond to **non-holiday weeks**, while **7%** occur during holiday periods.
- Store types are distributed as **22 Type A**, **17 Type B**, and **6 Type C** stores, with store sizes ranging from **34,875** to **219,622 square feet**.

---

#### Executive Summary

The Data Understanding phase provided a comprehensive overview of the Walmart Sales Forecasting dataset and confirmed that it is well structured for predictive modeling. The project integrates historical sales transactions, store characteristics, and external factors—including promotions, weather, fuel prices, holidays, and economic indicators—to support accurate weekly sales forecasting.

The data quality assessment revealed that the **training**, **testing**, and **stores** datasets contain no missing values or duplicate records. Missing values are limited to the **features** dataset and are primarily associated with promotional markdown variables, which likely reflect weeks when no promotional campaigns were active rather than data collection errors.

The exploratory analysis also highlighted several important business characteristics. Weekly sales exhibit considerable variability across stores and departments, with a right-skewed distribution and a small number of extremely high sales observations. The presence of negative sales values suggests returns or accounting adjustments that should be investigated during preprocessing. In addition, the dataset covers nearly three years of historical observations, providing sufficient information to capture seasonality, holiday effects, and long-term sales trends.

Overall, the dataset demonstrates strong data quality and a well-defined relational structure. After appropriate preprocessing—including date conversion, handling missing values, and merging the datasets—it will be well prepared for exploratory data analysis, feature engineering, and the development of machine learning and time-series forecasting models.